# Currency Agent A2A Example

This notebook demonstrates how to use the NeMo Agent Toolkit SDK to connect to a third-party A2A service (the LangGraph-based currency agent) for currency conversions.

## Key Features

1. **A2A Client Integration** - Connect to external A2A agents
2. **MCP Client Integration** - Use MCP servers for time operations
3. **Hybrid Tool Architecture** - Combine A2A and MCP tools

## Prerequisites

1. Start the external LangGraph currency agent:
   ```bash
   # Clone and run the a2a-samples repository
   cd external
   git clone https://github.com/a2aproject/a2a-samples.git
   cd a2a-samples/samples/python/agents/langgraph
   uv run app --port 11000
   ```

2. Set environment variables:
   - `NVIDIA_API_KEY` - NVIDIA API key
   - `GOOGLE_API_KEY` - Google Gemini API key (for currency agent)


In [ ]:
import os
import sys

# Add src to path for development
module_path = os.path.abspath('../../../src/')
if module_path not in sys.path:
    sys.path.insert(0, module_path)


In [ ]:
import logging

logger = logging.getLogger()
logger.setLevel(logging.INFO)


## Creating the Workflow

We'll create a workflow that combines:
- An A2A client to connect to the external currency agent
- An MCP client to get current time information


In [ ]:
from datetime import timedelta
from pathlib import Path

from pydantic import HttpUrl

from nat.agent.sdk import NatReActAgent
from nat.llm.sdk import NimLLM
from nat.plugins.a2a.sdk import A2AClient
from nat.plugins.mcp.client_config import MCPServerConfig
from nat.plugins.mcp.client_config import MCPToolOverrideConfig
from nat.plugins.mcp.sdk import MCPClient
from nat.utils.sdk.nat_workflow import NatWorkflow

# Create the LLM
llm = NimLLM(
    model_name="meta/llama-3.1-70b-instruct",
    temperature=0.0,
    max_tokens=1024,
    name="nim_llm",
)

# Create A2A client to connect to the currency agent
# Make sure the external currency agent is running on port 11000
currency_agent = A2AClient(
    url=HttpUrl("http://localhost:11000"),
    task_timeout=timedelta(seconds=60),
    name="currency_agent",
)

# Create MCP client for time operations
mcp_time = MCPClient(
    server=MCPServerConfig(
        transport="stdio",
        command="python",
        args=["-m", "mcp_server_time", "--local-timezone=America/Los_Angeles"],
    ),
    tool_overrides={
        "get_current_time": MCPToolOverrideConfig(
            alias="get_current_time_mcp_tool",
            description="Use this tool to get dates",
        ),
    },
    include=["get_current_time_mcp_tool"],
    name="mcp_date_time",
)

# Create the ReAct agent with both A2A and MCP tools
agent = NatReActAgent(
    tools=[currency_agent, mcp_time],
    llm=llm,
    verbose=True,
    parse_agent_response_max_retries=3,
)

# Wrap in NatWorkflow
nat_workflow = NatWorkflow(
    entrypoint=agent,
)

print("Workflow created successfully!")


## Saving the Configuration

Save the workflow configuration to a YAML file.


In [ ]:
config_dir = Path(os.getcwd()) / "config"
config_dir.mkdir(parents=True, exist_ok=True)

config_path = config_dir / "workflow_config.yaml"
nat_workflow.save_to_config_file(config_path)

print(f"Configuration saved to: {config_path}")
print("\n" + "="*50 + "\n")

with open(config_path) as f:
    print(f.read())


## Testing the Workflow

Test the workflow with a currency conversion query.

**Note**: The external currency agent must be running for this to work.


In [ ]:
# Test the workflow (requires external currency agent to be running)
# Uncomment to run:
# result = await nat_workflow.prompt(
#     "What was the USD to EUR exchange rate this day last year?"
# )
# print(result)


## Summary

This notebook demonstrated:

1. **A2AClient SDK Class** - Using `A2AClient` to connect to external A2A agents
2. **MCPClient SDK Class** - Using `MCPClient` to connect to MCP servers
3. **Hybrid Architecture** - Combining multiple tool sources in a single workflow
4. **Configuration Serialization** - Saving SDK-created workflows to YAML

### A2A Client Usage

```python
from nat.plugins.a2a.sdk import A2AClient

# Connect to any A2A-compatible agent
agent = A2AClient(
    url="http://localhost:11000",
    task_timeout=60,  # seconds
    name="my_agent",
)
```

### Related Examples

- [Math Assistant A2A](../math_assistant_a2a/) - NAT-to-NAT A2A with hybrid tools
